# scPoli 二级注释流程 Python版

基于一级校正结果 `cell_type_level1_corrected` 做 subset/recluster，并用 marker 得到二级注释列 `cell_type_level2`。
直接用原来的scpoli来

该文件的输出都在：/home/lixiangyu/zr/Annotate/ANNOTATE_new/8_annotate_level2/0514_rename_level2/output_allhuman/work_0514 

# import 

In [1]:
import anndata
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
sc.settings.verbosity=2
sc.settings.seed=1234
np.random.seed(1234)

In [2]:
outdir="../../7.5_correct/0521_no_Basophil/output_human/"
adata = sc.read_h5ad(os.path.join(outdir,"scPoli_concat_corrected_all_umap.h5ad"))
adata

AnnData object with n_obs × n_vars = 1004364 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [3]:
adata.obs['cell_type_level1_corrected'].value_counts()

cell_type_level1_corrected
T cell                   367292
Macrophage               103342
Fibroblast                87187
B cell                    86282
Neutrophil                76509
Smooth muscle cell        71467
Endothelial cell          64931
Natural killer cell       44858
Dendritic cell            36704
Monocyte                  34221
Pericyte                  19047
Mast cell                  6967
Erythrocyte/Erythroid      5557
Name: count, dtype: int64

In [4]:
adata.obsm["X_scPoli"].shape

(1004364, 10)

In [7]:
##合并后先画一张全体细胞的 UMAP，检查 cell_type_level1_corrected 是否正常
outdir="../0521_no_Basophil/output_allhuman/"
if "X_umap" in adata.obsm and "cell_type_level1_corrected" in adata.obs:
    sc.pl.umap(adata,color="cell_type_level1_corrected",legend_loc="on data",frameon=False,size=1,show=False)
    plt.savefig(os.path.join(outdir,"cell_type_level1_corrected.pdf"),bbox_inches="tight")
    plt.close()
adata

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


AnnData object with n_obs × n_vars = 1004364 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap', 'cell_type_level1_corrected_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

# mac

In [2]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/mac/"

In [9]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [10]:
mac = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Macrophage"],
    outdir=outdir,
    prefix="mac_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Macrophage']
Original object: AnnData object with n_obs × n_vars = 1004364 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap', 'cell_type_level1_corrected_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'
Subset object: AnnData object with n_obs × n_vars = 103342 × 28868
    

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/mac/umap_mac_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/mac/umap_mac_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/mac/umap_mac_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     3513
1     3205
2     3044
3     3017
4     2940
5     2893
6     2877
7     2812
8     2790
9     2773
10    2746
11    2741
12    2698
13    2635
14    2629
15    2623
16    2596
17    2504
18    2484
19    2464
20    2427
21    2391
22    2328
23    2319
24    2303
25    2276
26    2184
27    2183
28    2164
29    2099
30    1942
31    1889
32    1882
33    1867
34    1836
35    1671
36    1649
37    1534
38    1441
39    1381
40    1359
41    1353
42    1260
43    1252
44    1188
45    1159
46      21
Name: count, dtype: int64
Saved to: ./output_allhuman/mac/mac_level2_scPoli_recluster_umap.h5ad


In [3]:
work = sc.read_h5ad(os.path.join(outdir, "mac_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 103342 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [4]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
    "Inflammatory macrophage": ['S100A8', 'IL1B', 'S100A9','IRF7', 'IFITM3', 'ISG15', 'IFIT2'],
    "Foamy macrophage": ['TREM2', 'MARCO', 'FABP4','FABP5', 'CD36'],
    "Homeostatic/Resident macrophage": ['LYVE1', 'MRC1', 'F13A1', 'SEPP1', 'IGF1', 'GAS6', 'MERTK', 'STAB1', 'C1QA', 'C1QB', 'C1QC', 'FOLR2'],
    "Proliferative macrophage": ['MKI67', 'TOP2A', 'PCNA', 'BIRC5', 'MCM2','MCM4','MCM5', 'TK1'],
    "Matrix-remodeling/SMC-like macrophage": ['AEBP1', 'COL1A1', 'COL1A2', 'THBS2', 'MMP3', 'S100A16', 'TNC', 'PDGFRB'],
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mac_mono_neu_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mac_mono_neu_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
Inflammatory macrophage ['S100A8', 'IL1B', 'S100A9', 'IRF7', 'IFITM3', 'ISG15', 'IFIT2']
Foamy macrophage ['TREM2', 'MARCO', 'FABP4', 'FABP5', 'CD36']
Homeostatic/Resident macrophage ['LYVE1', 'MRC1', 'F13A1', 'IGF1', 'GAS6', 'MERTK', 'STAB1', 'C1QA', 'C1QB', 'C1QC', 'FOLR2']
Proliferative macrophage ['MKI67', 'TOP2A', 'PCNA', 'BIRC5', 'MCM2', 'MCM4', 'MCM5', 'TK1']
Matrix-remodeling/SMC-like macrophage ['AEBP1', 'COL1A1', 'COL1A2', 'THBS2', 'MMP3', 'S100A16', 'TNC', 'PDGFRB']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [5]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "mac_mono_neu_cluster_level2_marker_summary.csv"))


                            major_marker  major_marker_frac  n_cells              cluster_label_clean
cluster                                                                                              
0                       Foamy macrophage           0.806433     3513                 Foamy macrophage
1                Inflammatory macrophage           0.697660     3205          Inflammatory macrophage
2        Homeostatic/Resident macrophage           0.992116     3044  Homeostatic/Resident macrophage
3        Homeostatic/Resident macrophage           0.783560     3017  Homeostatic/Resident macrophage
4                       Foamy macrophage           0.687075     2940                 Foamy macrophage
5                Inflammatory macrophage           0.566194     2893          Inflammatory macrophage
6        Homeostatic/Resident macrophage           0.996872     2877  Homeostatic/Resident macrophage
7                       Foamy macrophage           0.616287     2812              

In [9]:
corrected_annotation = {
    "0": "Foamy macrophage",
    "1": "Inflammatory macrophage",
    "2": "Homeostatic/Resident macrophage",
    "3": "Homeostatic/Resident macrophage",
    "4": "Foamy macrophage",
    "5": "Inflammatory macrophage",
    "6": "Homeostatic/Resident macrophage",
    "7": "Foamy macrophage",
    "8": "Inflammatory macrophage",
    "9": "Inflammatory macrophage",
    "10": "Homeostatic/Resident macrophage",
    "11": "Homeostatic/Resident macrophage",
    "12": "Homeostatic/Resident macrophage",
    "13": "Homeostatic/Resident macrophage",
    "14": "Homeostatic/Resident macrophage",
    "15": "Homeostatic/Resident macrophage",
    "16": "Homeostatic/Resident macrophage",
    "17": "Homeostatic/Resident macrophage",
    "18": "Inflammatory macrophage",
    "19": "Homeostatic/Resident macrophage",
    "20": "Homeostatic/Resident macrophage",
    "21": "Homeostatic/Resident macrophage",
    "22": "Homeostatic/Resident macrophage",
    "23": "Homeostatic/Resident macrophage",
    "24": "Homeostatic/Resident macrophage",
    "25": "Homeostatic/Resident macrophage",
    "26": "Foamy macrophage",
    "27": "Inflammatory macrophage",
    "28": "Homeostatic/Resident macrophage",
    "29": "Homeostatic/Resident macrophage",
    "30": "Homeostatic/Resident macrophage",
    "31": "Inflammatory macrophage",
    "32": "Homeostatic/Resident macrophage",
    "33": "Homeostatic/Resident macrophage",
    "34": "Homeostatic/Resident macrophage",
    "35": "Homeostatic/Resident macrophage",
    "36": "Inflammatory macrophage",
    "37": "Inflammatory macrophage",
    "38": "Homeostatic/Resident macrophage",
    "39": "Inflammatory macrophage",
    "40": "Homeostatic/Resident macrophage",
    "41": "Homeostatic/Resident macrophage",
    "42": "Homeostatic/Resident macrophage",
    "43": "Homeostatic/Resident macrophage",
    "44": "Homeostatic/Resident macrophage",
    "45": "Homeostatic/Resident macrophage",
    "46": "Inflammatory macrophage"
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
Homeostatic/Resident macrophage    69091
Inflammatory macrophage            22802
Foamy macrophage                   11449
Name: count, dtype: int64


In [10]:
work.write_h5ad(os.path.join(outdir, "mac_scPoli_recluster_umap_level2.h5ad"))

In [11]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_mac_level2_cell_type_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_mac_level2_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_mac_level2_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_mac_level2_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_mac_level2_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

# smc

In [12]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/smc"

In [ ]:
# adata = sc.read_h5ad("./output_allhuman/work_0513_2/scPoli_concat_level2_marker_all.h5ad")
# adata

In [12]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [13]:
smc = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Smooth muscle cell"],
    outdir=outdir,
    prefix="smc_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Smooth muscle cell']
Original object: AnnData object with n_obs × n_vars = 1004364 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap', 'cell_type_level1_corrected_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'
Subset object: AnnData object with n_obs × n_vars = 71467 × 288

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/smc/umap_smc_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/smc/umap_smc_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/smc/umap_smc_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     2816
1     2594
2     2358
3     2166
4     2141
5     2126
6     2093
7     2060
8     2043
9     1994
10    1961
11    1919
12    1903
13    1880
14    1876
15    1847
16    1845
17    1797
18    1770
19    1765
20    1748
21    1716
22    1714
23    1711
24    1691
25    1641
26    1597
27    1539
28    1481
29    1476
30    1476
31    1462
32    1458
33    1405
34    1361
35    1254
36     991
37     980
38     824
39     818
40     769
41     744
42     550
43      79
44      28
Name: count, dtype: int64
Saved to: ./output_allhuman/smc/smc_level2_scPoli_recluster_umap.h5ad


In [13]:
work = sc.read_h5ad(os.path.join(outdir, "smc_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 71467 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [14]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
    "Smooth muscle cell": ['MYH11','MYL9','TPM2','CALD1','TAGLN','APOE','APOC1','AGT','NOTCH3','PDGFRB','MFAP4'],#'LUM','TNFRSF11B','ACTA2',
    "Fibromyocyte": ['FN1','LUM','TNFRSF11B','ACTA2','TCF21']
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_smc_fibro_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_smc_fibro_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
Smooth muscle cell ['MYH11', 'MYL9', 'TPM2', 'CALD1', 'TAGLN', 'APOE', 'APOC1', 'AGT', 'NOTCH3', 'PDGFRB', 'MFAP4']
Fibromyocyte ['FN1', 'LUM', 'TNFRSF11B', 'ACTA2', 'TCF21']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [15]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "smc_fibro_cluster_level2_marker_summary.csv"))


               major_marker  major_marker_frac  n_cells cluster_label_clean
cluster                                                                    
0        Smooth muscle cell           0.716974     2816  Smooth muscle cell
1        Smooth muscle cell           0.978026     2594  Smooth muscle cell
2        Smooth muscle cell           0.934266     2358  Smooth muscle cell
3        Smooth muscle cell           0.978763     2166  Smooth muscle cell
4              Fibromyocyte           0.694068     2141        Fibromyocyte
5        Smooth muscle cell           0.599718     2126  Smooth muscle cell
6        Smooth muscle cell           0.983755     2093  Smooth muscle cell
7        Smooth muscle cell           0.584951     2060  Smooth muscle cell
8        Smooth muscle cell           0.960842     2043  Smooth muscle cell
9        Smooth muscle cell           0.994483     1994  Smooth muscle cell
10             Fibromyocyte           0.583376     1961        Fibromyocyte
11       Smo

In [16]:
corrected_annotation = {
    "36": "Fibromyocyte",
    "43": "Smooth muscle cell"
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

## 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
Smooth muscle cell    57419
Fibromyocyte          14048
Name: count, dtype: int64


In [17]:
work.write_h5ad(os.path.join(outdir, "smc_scPoli_recluster_umap_level2.h5ad"))

In [18]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_smc_fib_cell_type_level2.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_smc_fib_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_smc_fib_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_smc_fib_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_smc_fib_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

# T cell

In [19]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/tc"

In [15]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [16]:
tc = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["T cell"],
    outdir=outdir,
    prefix="tc_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['T cell']
Original object: AnnData object with n_obs × n_vars = 1004364 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap', 'cell_type_level1_corrected_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'


Subset object: AnnData object with n_obs × n_vars = 367292 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap', 'cell_type_level1_corrected_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'
cell_type_level1_corrected.value_counts() after subsetting:
cell_type_level1_corrected
T cell    367292
Name

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/tc/umap_tc_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/tc/umap_tc_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/tc/umap_tc_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     11264
1     10458
2     10059
3      9520
4      9187
5      9175
6      9120
7      9021
8      8980
9      8952
10     8850
11     8587
12     8480
13     8399
14     8224
15     8154
16     7975
17     7891
18     7735
19     7642
20     7554
21     7521
22     7470
23     7470
24     7362
25     7344
26     7148
27     7068
28     6901
29     6796
30     6722
31     6605
32     6332
33     6183
34     6107
35     5865
36     5762
37     5671
38     5618
39     5522
40     5455
41     5439
42     5267
43     5027
44     4879
45     4856
46     4623
47     4163
48     3922
49     3796
50     3653
51     3560
52     1033
53      655
54      270
Name: count, dtype: int64
Saved to: ./output_allhuman/tc/tc_level2_scPoli_recluster_umap.h5ad


In [20]:
work = sc.read_h5ad(os.path.join(outdir, "tc_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 367292 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [21]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
    "CD8 T cell":['CCL4L2', 'CRTAM', 'GZMK', 'CCL4', 'CD8A', 'GZMH'], 
    "CD4 T cell":['IL7R', 'ANXA1', 'FTH1', 'CD4', 'BATF', 'TNFRSF4', 'TNFRSF18']
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_tc_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_tc_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
CD8 T cell ['CCL4L2', 'CRTAM', 'GZMK', 'CCL4', 'CD8A', 'GZMH']
CD4 T cell ['IL7R', 'ANXA1', 'FTH1', 'CD4', 'BATF', 'TNFRSF4', 'TNFRSF18']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [22]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "tc_cluster_level2_marker_summary.csv"))


        major_marker  major_marker_frac  n_cells cluster_label_clean
cluster                                                             
0         CD8 T cell           0.545543    11264          CD8 T cell
1         CD4 T cell           0.986613    10458          CD4 T cell
2         CD4 T cell           0.999404    10059          CD4 T cell
3         CD4 T cell           0.983403     9520          CD4 T cell
4         CD8 T cell           0.886688     9187          CD8 T cell
5         CD4 T cell           0.998474     9175          CD4 T cell
6         CD4 T cell           0.883882     9120          CD4 T cell
7         CD4 T cell           0.983594     9021          CD4 T cell
8         CD4 T cell           0.690646     8980          CD4 T cell
9         CD4 T cell           0.754357     8952          CD4 T cell
10        CD4 T cell           0.736836     8850          CD4 T cell
11        CD4 T cell           0.850006     8587          CD4 T cell
12        CD4 T cell           0.9

In [23]:
corrected_annotation = {
    "24" : "CD8 T cell"
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
CD4 T cell    247849
CD8 T cell    119443
Name: count, dtype: int64


In [24]:
work.write_h5ad(os.path.join(outdir, "tc_scPoli_recluster_umap_level2.h5ad"))

In [25]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_tc_cell_type_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_tc_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_tc_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_tc_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_tc_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

# ECs

In [26]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/ec"

In [18]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [19]:
ec = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Endothelial cell"],
    outdir=outdir,
    prefix="ec_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Endothelial cell']
Original object: AnnData object with n_obs × n_vars = 1004364 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap', 'cell_type_level1_corrected_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'
Subset object: AnnData object with n_obs × n_vars = 64931 × 28868

    finished (0:01:27)
running Leiden clustering
    finished (0:11:56)
computing UMAP
    finished (0:01:23)


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/ec/umap_ec_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/ec/umap_ec_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/ec/umap_ec_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     2355
1     2053
2     1844
3     1806
4     1791
5     1789
6     1784
7     1770
8     1765
9     1718
10    1717
11    1691
12    1662
13    1600
14    1593
15    1563
16    1533
17    1529
18    1523
19    1517
20    1458
21    1457
22    1445
23    1441
24    1434
25    1428
26    1411
27    1382
28    1368
29    1355
30    1297
31    1248
32    1229
33    1211
34    1208
35    1173
36    1167
37    1152
38     996
39     992
40     831
41     749
42     735
43     658
44     516
45     467
46     256
47     124
48      77
49      63
Name: count, dtype: int64
Saved to: ./output_allhuman/ec/ec_level2_scPoli_recluster_umap.h5ad


In [27]:
work = sc.read_h5ad(os.path.join(outdir, "ec_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 64931 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [28]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
    "Arterial endothelial cell": ['GJ4', 'GJ5', "GATA2", "MECOM", "GJA4", "GJA5"],
    "Lymphatic endothelial cell": ["LYVE1", "CCL21"], 
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_ec_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_ec_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
Arterial endothelial cell ['GATA2', 'MECOM', 'GJA4', 'GJA5']
Lymphatic endothelial cell ['LYVE1', 'CCL21']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [29]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "ec_cluster_level2_marker_summary.csv"))


                       major_marker  major_marker_frac  n_cells         cluster_label_clean
cluster                                                                                    
0        Lymphatic endothelial cell           0.626752     2355  Lymphatic endothelial cell
1        Lymphatic endothelial cell           0.622504     2053  Lymphatic endothelial cell
2        Lymphatic endothelial cell           0.535249     1844  Lymphatic endothelial cell
3         Arterial endothelial cell           0.521595     1806   Arterial endothelial cell
4         Arterial endothelial cell           0.520380     1791   Arterial endothelial cell
5         Arterial endothelial cell           0.867524     1789   Arterial endothelial cell
6        Lymphatic endothelial cell           0.522422     1784  Lymphatic endothelial cell
7         Arterial endothelial cell           0.850282     1770   Arterial endothelial cell
8        Lymphatic endothelial cell           0.620397     1765  Lymphatic endot

In [33]:
corrected_annotation = {
    "0": "Arterial endothelial cell",
    "1": "Arterial endothelial cell",
    "2": "Arterial endothelial cell",
    "3": "Arterial endothelial cell",
    "4": "Arterial endothelial cell",
    "5": "Arterial endothelial cell",
    "6": "Arterial endothelial cell",
    "7": "Arterial endothelial cell",
    "8": "Arterial endothelial cell",
    "9": "Arterial endothelial cell",
    "10": "Arterial endothelial cell",
    "11": "Arterial endothelial cell",
    "12": "Arterial endothelial cell",
    "13": "Arterial endothelial cell",
    "14": "Arterial endothelial cell",
    "15": "Arterial endothelial cell",
    "16": "Arterial endothelial cell",
    "17": "Arterial endothelial cell",
    "18": "Arterial endothelial cell",
    "19": "Arterial endothelial cell",
    "20": "Arterial endothelial cell",
    "21": "Arterial endothelial cell",
    "22": "Arterial endothelial cell",
    "23": "Arterial endothelial cell",
    "24": "Arterial endothelial cell",
    "25": "Arterial endothelial cell",
    "26": "Arterial endothelial cell",
    "27": "Arterial endothelial cell",
    "28": "Arterial endothelial cell",
    "29": "Arterial endothelial cell",
    "30": "Arterial endothelial cell",
    "31": "Arterial endothelial cell",
    "32": "Arterial endothelial cell",
    "33": "Arterial endothelial cell",
    "34": "Arterial endothelial cell",
    "35": "Arterial endothelial cell",
    "36": "Arterial endothelial cell",
    "37": "Arterial endothelial cell",
    "38": "Arterial endothelial cell",
    "39": "Arterial endothelial cell",
    "40": "Lymphatic endothelial cell",
    "41": "Arterial endothelial cell",
    "42": "Arterial endothelial cell",
    "43": "Arterial endothelial cell",
    "44": "Arterial endothelial cell",
    "45": "Arterial endothelial cell",
    "46": "Other endothelial cell",
    "47": "Arterial endothelial cell",
    "48": "undefine",
    "49": "Arterial endothelial cell"
   
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
Arterial endothelial cell     63767
Lymphatic endothelial cell      831
Other endothelial cell          256
undefine                         77
Name: count, dtype: int64


In [34]:
work.write_h5ad(os.path.join(outdir, "ec_scPoli_recluster_umap_level2.h5ad"))

In [35]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_ec_cell_type_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_ec_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_ec_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_ec_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_ec_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

# dc

In [44]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/dc"

In [21]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [22]:
dc = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Dendritic cell"],
    outdir=outdir,
    prefix="dc_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Dendritic cell']
Original object: AnnData object with n_obs × n_vars = 1004364 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap', 'cell_type_level1_corrected_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'
Subset object: AnnData object with n_obs × n_vars = 36704 × 28868
 

    finished (0:00:49)
running Leiden clustering
    finished (0:09:22)
computing UMAP
    finished (0:00:44)


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/dc/umap_dc_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/dc/umap_dc_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/dc/umap_dc_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     1455
1     1354
2     1315
3     1304
4     1295
5     1280
6     1274
7     1232
8     1230
9     1205
10    1194
11    1189
12    1166
13    1161
14    1145
15    1036
16    1025
17    1004
18     963
19     932
20     907
21     905
22     886
23     883
24     800
25     791
26     790
27     785
28     768
29     765
30     723
31     687
32     626
33     569
34     563
35     492
36     424
37     361
38     180
39      40
Name: count, dtype: int64
Saved to: ./output_allhuman/dc/dc_level2_scPoli_recluster_umap.h5ad


In [45]:
work = sc.read_h5ad(os.path.join(outdir, "dc_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 36704 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [46]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
    # "cDC1": ["CLEC9A", "IRF8", "SNX3"],
    # "cDC2": ["CD1C", "CLEC10A", "FCER1A"],
    # "Plasmacytoid dendritic cell": ["IRF7", "TLR7", "TLR9", "NRP1", "CLEC4C", "SCAMP5", "GZMB", "TCF4"]
    "cDC1": ["CLEC9A", "XCR1", "CADM1", "BATF3", "IRF8", "THBD"],
    "cDC2": ["CD1C", "CLEC10A", "FCER1A", "CD1E", "FCGR2B", "SIRPA"],
    "Plasmacytoid dendritic cell": ["CLEC4C", "IL3RA", "LILRA4", "TCF4", "IRF7", "GZMB", "JCHAIN", "SCT"]
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_dc_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_dc_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
cDC1 ['CLEC9A', 'XCR1', 'CADM1', 'BATF3', 'IRF8', 'THBD']
cDC2 ['CD1C', 'CLEC10A', 'FCER1A', 'CD1E', 'FCGR2B', 'SIRPA']
Plasmacytoid dendritic cell ['CLEC4C', 'IL3RA', 'LILRA4', 'TCF4', 'IRF7', 'GZMB', 'JCHAIN', 'SCT']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [47]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "dc_cluster_level2_marker_summary.csv"))


                        major_marker  major_marker_frac  n_cells          cluster_label_clean
cluster                                                                                      
0                               cDC2           0.610309     1455                         cDC2
1                               cDC2           0.953471     1354                         cDC2
2                               cDC2           0.517871     1315                         cDC2
3        Plasmacytoid dendritic cell           0.653374     1304  Plasmacytoid dendritic cell
4                               cDC2           0.715830     1295                         cDC2
5                               cDC2           0.796094     1280                         cDC2
6                               cDC2           0.696232     1274                         cDC2
7                               cDC2           0.533279     1232                         cDC2
8        Plasmacytoid dendritic cell           0.846341     

In [57]:
corrected_annotation = {
    "19": "cDC2",
    "18": "cDC1",
    "20": "cDC2",
    "27": "cDC1",
    "31": "Plasmacytoid dendritic cell",
    "32": "cDC2",
    "33": "cDC1",
    "36": "cDC1",
    "39": "undefine",
    
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
cDC2                           28914
Plasmacytoid dendritic cell     5009
cDC1                            2741
undefine                          40
Name: count, dtype: int64


In [58]:
work.write_h5ad(os.path.join(outdir, "dc_scPoli_recluster_umap_level2.h5ad"))

In [59]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_dc_cell_type_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_nk_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_dc_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_dc_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_nk_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

# B cells

In [2]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/bc"

In [24]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [25]:
bc = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["B cell"],
    outdir=outdir,
    prefix="bc_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['B cell']
Original object: AnnData object with n_obs × n_vars = 1004364 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap', 'cell_type_level1_corrected_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'
Subset object: AnnData object with n_obs × n_vars = 86282 × 28868
    obs: 

normalizing counts per cell
    finished (0:00:06)
lognorm min 0.4206185 max 8.99246 integer-like should be False False
cell_type_level1_corrected.value_counts() after normalization:
cell_type_level1_corrected
B cell    86282
Name: count, dtype: int64
Formal correction/reclustering:
Only using cell_type_level1_corrected, X_scPoli, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.
work object: AnnData object with n_obs × n_vars = 86282 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/bc/umap_bc_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/bc/umap_bc_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/bc/umap_bc_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     3412
1     3343
2     3184
3     2683
4     2626
5     2550
6     2453
7     2417
8     2365
9     2348
10    2337
11    2318
12    2224
13    2186
14    2130
15    2051
16    1985
17    1925
18    1900
19    1843
20    1777
21    1754
22    1731
23    1706
24    1705
25    1603
26    1574
27    1561
28    1560
29    1556
30    1492
31    1483
32    1433
33    1359
34    1356
35    1316
36    1254
37    1221
38    1212
39    1162
40    1143
41    1111
42    1107
43    1088
44     855
45     774
46     557
47     491
48     477
49     304
50     280
Name: count, dtype: int64
Saved to: ./output_allhuman/bc/bc_level2_scPoli_recluster_umap.h5ad


In [3]:
work = sc.read_h5ad(os.path.join(outdir, "bc_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 86282 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [4]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
   "B cell": ['CD79A', 'CD79B', 'MS4A1', 'CD22', 'FCER2'],#IGKC
    "Plasma cell": ['IGKC', 'IGHM', 'IGHA1', 'IGLC2', 'IGLC3', 'JCHAIN']
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_bc_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_bc_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
B cell ['CD79A', 'CD79B', 'MS4A1', 'CD22', 'FCER2']
Plasma cell ['IGKC', 'IGHM', 'IGHA1', 'IGLC2', 'IGLC3', 'JCHAIN']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [5]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "bc_cluster_level2_marker_summary.csv"))


        major_marker  major_marker_frac  n_cells cluster_label_clean
cluster                                                             
0             B cell           0.830891     3412              B cell
1        Plasma cell           0.999402     3343         Plasma cell
2        Plasma cell           0.992148     3184         Plasma cell
3        Plasma cell           0.997018     2683         Plasma cell
4             B cell           0.507616     2626              B cell
5             B cell           0.834118     2550              B cell
6             B cell           0.820628     2453              B cell
7             B cell           0.894911     2417              B cell
8             B cell           0.871882     2365              B cell
9             B cell           0.919080     2348              B cell
10       Plasma cell           0.620454     2337         Plasma cell
11            B cell           0.820966     2318              B cell
12            B cell           0.9

In [6]:
corrected_annotation = {
    "10": "B cell",
    "18": "B cell",
    "19": "B cell",
    "25": "B cell",
    "30": "B cell",
    "45": "B cell",
    "47": "B cell",
    "49": "B cell"
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
B cell         77072
Plasma cell     9210
Name: count, dtype: int64


In [7]:
work.write_h5ad(os.path.join(outdir, "bc_scPoli_recluster_umap_level2.h5ad"))

In [8]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_bc_cell_type_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_bc_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_bc_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_bc_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_bc_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

# mono

In [9]:
label_col = "cell_type_level1_corrected"
cluster_key = "leiden_scpoli_res3"
outdir="./output_allhuman/mono/"

In [27]:
def subset_normalize_recluster_scpoli(adata, target_cell_types, outdir, prefix=None, label_col="cell_type_level1_corrected", counts_layer="counts", lognorm_layer="lognorm", scpoli_rep="X_scPoli", cluster_key=None, n_neighbors=150, leiden_resolution=3, leiden_random_state=1234, umap_min_dist=1.1, umap_random_state=1234, metric="euclidean", target_sum=1e4, size=2, save_h5ad=True):
    os.makedirs(outdir, exist_ok=True)
    target_cell_types = list(target_cell_types)
    if prefix is None:
        prefix = "_".join([x.lower().replace(" ", "_").replace("/", "_").replace("+", "pos") for x in target_cell_types])
    if cluster_key is None:
        cluster_key = f"leiden_scpoli_res{leiden_resolution}"
    if label_col not in adata.obs:
        raise ValueError(f"{label_col} not found in adata.obs")
    if counts_layer not in adata.layers:
        raise ValueError(f"{counts_layer} not found in adata.layers")
    if scpoli_rep not in adata.obsm:
        raise ValueError(f"{scpoli_rep} not found in adata.obsm")

    print("Target cell types:", target_cell_types)
    print("Original object:", adata)
    work = adata[adata.obs[label_col].astype(str).isin(target_cell_types), :].copy()
    print("Subset object:", work)
    print(f"{label_col}.value_counts() after subsetting:")
    print(work.obs[label_col].value_counts())
    if work.n_obs == 0:
        raise ValueError("No cells selected. Please check target_cell_types and label_col.")

    work.X = work.layers[counts_layer].copy()
    work.layers[counts_layer] = work.X.copy()
    sc.pp.normalize_total(work, target_sum=target_sum)
    sc.pp.log1p(work)
    work.layers[lognorm_layer] = work.X.copy()

    x = work.layers[lognorm_layer]
    v = x.data if sp.issparse(x) else np.ravel(x)
    v = v[:min(len(v), 1000000)]
    print("lognorm min", np.min(v), "max", np.max(v), "integer-like should be False", np.allclose(v, np.round(v)))
    print(f"{label_col}.value_counts() after normalization:")
    print(work.obs[label_col].value_counts())

    print("Formal correction/reclustering:")
    print(f"Only using {label_col}, {scpoli_rep}, and marker-based level2 annotation; not using cell_type_ori / cell_type_level1 as ground truth.")
    print("work object:", work)
    print(f"{label_col}.value_counts() in work:")
    print(work.obs[label_col].value_counts())

    sc.pp.neighbors(work, use_rep=scpoli_rep, n_neighbors=n_neighbors, metric=metric)
    sc.tl.leiden(work, resolution=leiden_resolution, key_added=cluster_key, random_state=leiden_random_state)
    sc.tl.umap(work, min_dist=umap_min_dist, random_state=umap_random_state)

    for c in [cluster_key, label_col]:
        sc.pl.umap(work, color=c, legend_loc="on data", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_{c}.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    if "dataset" in work.obs:
        sc.pl.umap(work, color="dataset", frameon=False, size=size, show=False)
        fig_path = os.path.join(outdir, f"umap_{prefix}_dataset.pdf")
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        print(f"Saved UMAP: {fig_path}")

    print(f"{cluster_key}.value_counts().sort_index():")
    print(work.obs[cluster_key].value_counts().sort_index())

    if save_h5ad:
        save_path = os.path.join(outdir, f"{prefix}_scPoli_recluster_umap.h5ad")
        work.write_h5ad(save_path)
        print(f"Saved to: {save_path}")

    return work

In [28]:
mono = subset_normalize_recluster_scpoli(
    adata=adata,
    target_cell_types=["Monocyte"],
    outdir=outdir,
    prefix="mono_level2",
    cluster_key="leiden_scpoli_res3"
)

Target cell types: ['Monocyte']
Original object: AnnData object with n_obs × n_vars = 1004364 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap', 'cell_type_level1_corrected_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'
Subset object: AnnData object with n_obs × n_vars = 34221 × 28868
    obs

    finished (0:00:47)
running Leiden clustering
    finished (0:04:00)
computing UMAP
    finished (0:00:41)


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/mono/umap_mono_level2_leiden_scpoli_res3.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/mono/umap_mono_level2_cell_type_level1_corrected.pdf


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


Saved UMAP: ./output_allhuman/mono/umap_mono_level2_dataset.pdf
leiden_scpoli_res3.value_counts().sort_index():
leiden_scpoli_res3
0     1307
1     1239
2     1178
3     1156
4     1155
5     1133
6     1101
7     1096
8     1090
9     1087
10    1077
11    1075
12    1046
13     995
14     953
15     951
16     931
17     923
18     872
19     859
20     846
21     842
22     842
23     834
24     825
25     822
26     811
27     810
28     790
29     763
30     739
31     685
32     675
33     587
34     560
35     549
36     394
37     331
38     184
39     108
Name: count, dtype: int64
Saved to: ./output_allhuman/mono/mono_level2_scPoli_recluster_umap.h5ad


In [10]:
work = sc.read_h5ad(os.path.join(outdir, "mono_level2_scPoli_recluster_umap.h5ad"))
work

AnnData object with n_obs × n_vars = 34221 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_level1_corrected_colors', 'dataset_colors', 'leiden', 'leiden_scpoli_res3_colors', 'log1p', 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'lognorm', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [11]:
# 如果 h5ad 里 var_names 不是 gene symbol，先改成 gene symbol
work.var_names = work.var["original_gene_names"].astype(str).values
work.var_names_make_unique()

# marker-based sanity check：用 marker 给每个细胞打 marker_top 标签
marker_dict = {
   "Classical monocyte": ['CD14', 'FCN1', 'LST1', 'VCAN', 'CCR2', 'SELL', 'CTSS'],
    "Intermediate monocyte": [ 'MS4A7', 'HLA-DPA1', 'HLA-DPB1', 'ITGAX', 'IRF7', 'STAT1'],# CD14,FCGR3A,
    "Non-classical monocyte": ['FCGR3A', 'CX3CR1', 'NR4A1', 'ITGAL', 'PDE4B', 'LILRB1', 'HSPA1A']
}

marker_dict = {k: [g for g in v if g in work.var_names] for k, v in marker_dict.items()}
marker_dict = {k: v for k, v in marker_dict.items() if len(v) > 0}

print("marker genes used:")
for k, v in marker_dict.items():
    print(k, v)

def mean_expr_by_genes(adata, genes, layer="lognorm"):
    X = adata[:, genes].layers[layer] if layer in adata.layers else adata[:, genes].X
    return np.asarray(X.mean(axis=1)).ravel()

score_cols = []
score_col_to_label = {}

for ct, genes in marker_dict.items():
    safe = ct.replace(" ", "_").replace("/", "_")
    col = f"marker_score__{safe}"
    work.obs[col] = mean_expr_by_genes(work, genes, layer="lognorm")
    score_cols.append(col)
    score_col_to_label[col] = ct

if len(score_cols) > 0:
    score_mat = work.obs[score_cols].to_numpy(dtype=float)
    order = np.argsort(score_mat, axis=1)
    top_idx = order[:, -1]
    second_idx = order[:, -2] if len(score_cols) > 1 else order[:, -1]

    score_labels = np.array([score_col_to_label[c] for c in score_cols])
    work.obs["marker_top"] = score_labels[top_idx]
    work.obs["marker_top_score"] = score_mat[np.arange(work.n_obs), top_idx]
    work.obs["marker_second_score"] = score_mat[np.arange(work.n_obs), second_idx]
    work.obs["marker_margin"] = work.obs["marker_top_score"] - work.obs["marker_second_score"]
    work.obs["marker_disagree"] = ((work.obs["marker_top"].astype(str) != work.obs[label_col].astype(str)) & (work.obs["marker_margin"].astype(float) > 0.1))

    sc.pl.dotplot(work, var_names=marker_dict, groupby=label_col, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mono_by_level1_corrected.pdf"), bbox_inches="tight")
    plt.close()

    sc.pl.dotplot(work, var_names=marker_dict, groupby=cluster_key, standard_scale="var", show=False)
    plt.savefig(os.path.join(outdir, "dot_marker_mono_by_cluster.pdf"), bbox_inches="tight")
    plt.close()
else:
    work.obs["marker_top"] = "Unknown"
    work.obs["marker_margin"] = np.nan
    work.obs["marker_disagree"] = False
    print("no marker genes found; skip marker sanity check")

marker genes used:
Classical monocyte ['CD14', 'FCN1', 'LST1', 'VCAN', 'CCR2', 'SELL', 'CTSS']
Intermediate monocyte ['MS4A7', 'HLA-DPA1', 'HLA-DPB1', 'ITGAX', 'IRF7', 'STAT1']
Non-classical monocyte ['FCGR3A', 'CX3CR1', 'NR4A1', 'ITGAL', 'PDE4B', 'LILRB1', 'HSPA1A']


/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_bool.groupby(level=0).sum() / obs_bool.groupby(level=0).count()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dot_color_df = self.obs_tidy.groupby(level=0).mean()
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_dotplot.py:747: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  dot

In [12]:
# cluster-level 二级注释：不用预测标签多数派，只看每个 scPoli cluster 内的 marker_top 多数派
if "marker_top" in work.obs and (work.obs["marker_top"].astype(str) != "Unknown").any():
    marker_tab = pd.crosstab(work.obs[cluster_key].astype(str), work.obs["marker_top"].astype(str))
    marker_frac = marker_tab.div(marker_tab.sum(axis=1), axis=0)
    cluster_major_marker = marker_tab.idxmax(axis=1)
    cluster_major_marker_frac = marker_frac.max(axis=1)
    n_cells = marker_tab.sum(axis=1)
else:
    clusters = work.obs[cluster_key].astype(str).value_counts().sort_index().index
    cluster_major_marker = pd.Series("Unknown", index=clusters)
    cluster_major_marker_frac = pd.Series(np.nan, index=clusters)
    n_cells = work.obs[cluster_key].astype(str).value_counts().reindex(clusters).fillna(0).astype(int)

cluster_summary = pd.DataFrame({
    "cluster": cluster_major_marker.index.astype(str),
    "major_marker": cluster_major_marker.values,
    "major_marker_frac": cluster_major_marker_frac.reindex(cluster_major_marker.index).values,
    "n_cells": n_cells.reindex(cluster_major_marker.index).values,
}).set_index("cluster")

# marker-only 结果：不再使用预测标签多数派
cluster_summary["cluster_label_clean"] = cluster_summary["major_marker"].astype(str)
cluster_summary.loc[cluster_summary["cluster_label_clean"].eq("Unknown"), "cluster_label_clean"] = "Uncertain"

cluster_summary = cluster_summary.copy()
cluster_summary["cluster_num"] = cluster_summary.index.astype(int)
cluster_summary = cluster_summary.sort_values("cluster_num")

print(cluster_summary.drop(columns="cluster_num").to_string())
cluster_summary.drop(columns="cluster_num").to_csv(os.path.join(outdir, "mono_cluster_level2_marker_summary.csv"))


                  major_marker  major_marker_frac  n_cells    cluster_label_clean
cluster                                                                          
0           Classical monocyte           0.920428     1307     Classical monocyte
1           Classical monocyte           0.686844     1239     Classical monocyte
2           Classical monocyte           0.923599     1178     Classical monocyte
3           Classical monocyte           0.811419     1156     Classical monocyte
4           Classical monocyte           0.690909     1155     Classical monocyte
5           Classical monocyte           0.789938     1133     Classical monocyte
6           Classical monocyte           0.964578     1101     Classical monocyte
7           Classical monocyte           0.846715     1096     Classical monocyte
8           Classical monocyte           0.991743     1090     Classical monocyte
9           Classical monocyte           0.946642     1087     Classical monocyte
10          Clas

In [18]:
corrected_annotation = {
    "14": "Classical monocyte",
    "39": "Intermediate monocyte",
}
# marker-only 二级注释结果：不再写回/覆盖一级列 cell_type_level1_corrected
level2_col = "cell_type_level2_marker"
cluster_clean_map = cluster_summary["cluster_label_clean"].to_dict()
work.obs[level2_col] = work.obs[cluster_key].astype(str).map(cluster_clean_map).fillna("Unknown")

# 如需人工覆盖 marker-only 结果，再取消下面几行注释。
manual_map = work.obs[cluster_key].astype(str).map(corrected_annotation)
manual_mask = manual_map.notna()
work.obs.loc[manual_mask, level2_col] = manual_map.loc[manual_mask]

print(work.obs[level2_col].value_counts())

cell_type_level2_marker
Classical monocyte       31679
Intermediate monocyte     2542
Name: count, dtype: int64


In [19]:
work.write_h5ad(os.path.join(outdir, "mono_scPoli_recluster_umap_level2.h5ad"))

In [20]:
# 注释后 UMAP：看最终 corrected label
sc.pl.umap(
    work,
    color="cell_type_level2_marker",
    legend_loc="on data",
    frameon=False,
    size=2,
    show=False
)
plt.savefig(os.path.join(outdir, "umap_mono_cell_type_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 UMAP：和 cluster 对照
# sc.pl.umap(
#     work,
#     color=cluster_key,
#     legend_loc="on data",
#     frameon=False,
#     size=2,
#     show=False
# )
# plt.savefig(os.path.join(outdir, "umap_mono_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

# 注释后 UMAP：如果有 dataset，也看 batch/dataset 分布
if "dataset" in work.obs:
    sc.pl.umap(
        work,
        color="dataset",
        frameon=False,
        size=2,
        show=False
    )
    plt.savefig(os.path.join(outdir, "umap_mono_dataset_after_correction.pdf"), bbox_inches="tight")
    plt.close()

# 注释后 dotplot：按最终 corrected label 分组
sc.pl.dotplot(
    work,
    var_names=marker_dict,
    groupby="cell_type_level2_marker",
    standard_scale="var",
    show=False
)
plt.savefig(os.path.join(outdir, "dot_marker_mono_by_level2_marker.pdf"), bbox_inches="tight")
plt.close()

# # 注释后 dotplot：按 cluster 分组，方便检查每个 cluster 的 marker
# sc.pl.dotplot(
#     work,
#     var_names=marker_dict,
#     groupby=cluster_key,
#     standard_scale="var",
#     show=False
# )
# plt.savefig(os.path.join(outdir, "dot_marker_mono_by_cluster_after_correction.pdf"), bbox_inches="tight")
# plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categoric

In [ ]:
####还有mast cell,先不做

# 合并

In [2]:
outdir="../../7.5_correct/0521_no_Basophil/output_human/"
adata = sc.read_h5ad(os.path.join(outdir,"scPoli_concat_corrected_all_umap.h5ad"))
adata

AnnData object with n_obs × n_vars = 1004364 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [3]:
adata.obs["cell_type_level1_corrected"].value_counts()

cell_type_level1_corrected
T cell                   367292
Macrophage               103342
Fibroblast                87187
B cell                    86282
Neutrophil                76509
Smooth muscle cell        71467
Endothelial cell          64931
Natural killer cell       44858
Dendritic cell            36704
Monocyte                  34221
Pericyte                  19047
Mast cell                  6967
Erythrocyte/Erythroid      5557
Name: count, dtype: int64

In [5]:
outdir="./output_allhuman/"
adata_bc = sc.read_h5ad(os.path.join(outdir, "bc/bc_scPoli_recluster_umap_level2.h5ad"))
adata_dc = sc.read_h5ad(os.path.join(outdir, "dc/dc_scPoli_recluster_umap_level2.h5ad"))
adata_mac = sc.read_h5ad(os.path.join(outdir, "mac/mac_scPoli_recluster_umap_level2.h5ad"))
adata_smc = sc.read_h5ad(os.path.join(outdir, "smc/smc_scPoli_recluster_umap_level2.h5ad"))
adata_tc = sc.read_h5ad(os.path.join(outdir, "tc/tc_scPoli_recluster_umap_level2.h5ad"))
adata_mono = sc.read_h5ad(os.path.join(outdir, "mono/mono_scPoli_recluster_umap_level2.h5ad"))
adata_ec = sc.read_h5ad(os.path.join(outdir, "ec/ec_scPoli_recluster_umap_level2.h5ad"))

In [6]:
adata_list = [adata_bc, adata_dc,adata_mac,adata_smc,adata_tc,adata_ec,adata_mono]

In [7]:
adata_concat = anndata.concat(adata_list, join='outer', fill_value=0.0)

In [8]:
adata_concat.write(os.path.join(outdir,"scPoli_concat_level2.h5ad"))
adata_concat

AnnData object with n_obs × n_vars = 764239 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3', 'marker_score__B_cell', 'marker_score__Plasma_cell', 'marker_top', 'marker_top_score', 'marker_second_score', 'marker_margin', 'marker_disagree', 'cell_type_level2_marker', 'marker_score__cDC1', 'marker_score__cDC2', 'marker_score__Plasmacytoid_dendritic_cell', 'marker_score__Inflammatory_macrophage', 'marker_score__Foamy_macrophage', 'marker_score__Homeostatic_Resident_m

In [4]:
outdir="./output_allhuman/"
adata_concat = sc.read_h5ad(os.path.join(outdir,"scPoli_concat_level2.h5ad"))
adata_concat

AnnData object with n_obs × n_vars = 764239 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'leiden_scpoli_res3', 'marker_score__B_cell', 'marker_score__Plasma_cell', 'marker_top', 'marker_top_score', 'marker_second_score', 'marker_margin', 'marker_disagree', 'cell_type_level2_marker', 'marker_score__cDC1', 'marker_score__cDC2', 'marker_score__Plasmacytoid_dendritic_cell', 'marker_score__Inflammatory_macrophage', 'marker_score__Foamy_macrophage', 'marker_score__Homeostatic_Resident_m

In [5]:
adata_concat.obs_names

Index(['1_JD__AAACCCAAGGAGTCTG-1', '1_JD__AAACCCATCCTTCGAC-1',
       '1_JD__AAACGAAAGCGTCTCG-1', '1_JD__AAACGAACACTTCCTG-1',
       '1_JD__AAACGCTAGCATCCTA-1', '1_JD__AAACGCTCAAACTAAG-1',
       '1_JD__AAAGAACGTGACACAG-1', '1_JD__AAAGAACTCCGCGAGT-1',
       '1_JD__AAAGAACTCTCAATCT-1', '1_JD__AAAGGGCTCTCGAGTA-1',
       ...
       'GSE196943_11__TCGATTTGTCAAGCGA_K_blood',
       'GSE196943_11__TCTACATGTCAGGCAA_K_blood',
       'GSE196943_11__TGCCGAGTCGTAGGAG_K_blood',
       'GSE196943_11__TTCACCGTCCATTTCA_K_plaque',
       'GSE196943_11__ACTGCAATCAATCAGC_K_plaque',
       'GSE196943_11__CCCATTGGTATTTCCT_K_plaque',
       'GSE196943_10__ATCATTCCATTGGGAG_J_blood',
       'GSE196943_10__GACCCTTGTGTTACAC_J_blood',
       'GSE196943_10__GTTCCGTTCGTTGCCT_J_blood',
       'GSE196943_7__GTTGTGACAGCGTTTA_G_plaque'],
      dtype='object', name='match_id', length=764239)

In [6]:
# Extract barcodes and cell_type_level2_marker values
barcodes = adata_concat.obs_names
cell_types_level2 = adata_concat.obs["cell_type_level2_marker"]

# Create the mapping
mapping = dict(zip(barcodes, cell_types_level2))

In [7]:
adata.obs_names

Index(['1_JD__AAACCCAAGAGGTTAT-1', '1_JD__AAACCCAAGCAACAAT-1',
       '1_JD__AAACCCAAGGAGTCTG-1', '1_JD__AAACCCACAACTCATG-1',
       '1_JD__AAACCCACAGCTCTGG-1', '1_JD__AAACCCAGTCAAGGCA-1',
       '1_JD__AAACCCAGTCAATCTG-1', '1_JD__AAACCCAGTCACCCTT-1',
       '1_JD__AAACCCAGTGCATTAC-1', '1_JD__AAACCCAGTGCGGCTT-1',
       ...
       'GSE196943_8__TTTGGTTGTCGCATAT_H_plaque',
       'GSE196943_8__ATAACGCAGTACGCCC_H_plaque',
       'GSE196943_8__CCATTCGAGTAGCCGA_H_plaque',
       'GSE196943_8__GAATAAGAGAAAGTGG_H_plaque',
       'GSE196943_8__GACCAATCATGGTAGG_H_plaque',
       'GSE196943_8__GACCTGGGTCTTCAAG_H_plaque',
       'GSE196943_8__TGGGCGTTCTTCTGGC_H_plaque',
       'GSE196943_8__GAGCAGATCAGAGGTG_H_plaque',
       'GSE196943_8__GAGTCCGGTTCTGGTA_H_plaque',
       'GSE196943_8__TCATTTGAGTTGTCGT_H_plaque'],
      dtype='object', name='match_id', length=1004364)

In [8]:
# init new column
adata.obs["cell_type_level2"] = "no map"

In [9]:
### more fast
if "cell_type_level2" not in adata.obs.columns:
    adata.obs["cell_type_level2"] = pd.NA

mapped = adata.obs.index.to_series().map(mapping)
mask = mapped.notna()
adata.obs.loc[mask, "cell_type_level2"] = mapped[mask].to_numpy()

In [10]:
adata.obs["cell_type_level2"].value_counts()

cell_type_level2
CD4 T cell                         247849
no map                             240125
CD8 T cell                         119443
B cell                              77072
Homeostatic/Resident macrophage     69091
Arterial endothelial cell           63767
Smooth muscle cell                  57419
Classical monocyte                  31679
cDC2                                28914
Inflammatory macrophage             22802
Fibromyocyte                        14048
Foamy macrophage                    11449
Plasma cell                          9210
Plasmacytoid dendritic cell          5009
cDC1                                 2741
Intermediate monocyte                2542
Lymphatic endothelial cell            831
Other endothelial cell                256
undefine                              117
Name: count, dtype: int64

In [11]:
adata = adata[adata.obs['cell_type_level2'] != 'undefine'].copy()
adata

AnnData object with n_obs × n_vars = 1004247 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'cell_type_level2'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [12]:
mask = adata.obs["cell_type_level2"] == "no map"
adata.obs.loc[mask, "cell_type_level2"] = adata.obs.loc[mask, "cell_type_level1_corrected"]

In [13]:
adata.obs["cell_type_level2"].value_counts()

cell_type_level2
CD4 T cell                         247849
CD8 T cell                         119443
Fibroblast                          87187
B cell                              77072
Neutrophil                          76509
Homeostatic/Resident macrophage     69091
Arterial endothelial cell           63767
Smooth muscle cell                  57419
Natural killer cell                 44858
Classical monocyte                  31679
cDC2                                28914
Inflammatory macrophage             22802
Pericyte                            19047
Fibromyocyte                        14048
Foamy macrophage                    11449
Plasma cell                          9210
Mast cell                            6967
Erythrocyte/Erythroid                5557
Plasmacytoid dendritic cell          5009
cDC1                                 2741
Intermediate monocyte                2542
Lymphatic endothelial cell            831
Other endothelial cell                256
Name: count, dtyp

In [14]:
adata.write(os.path.join(outdir,"scPoli_concat_level2_marker_allhuman.h5ad"))
adata

AnnData object with n_obs × n_vars = 1004247 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'cell_type_level2'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [15]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/ANNOTATE_new/8_annotate_level2/0521_no_Basophil/output_allhuman/scPoli_concat_level2_marker_allhuman.h5ad")
adata

AnnData object with n_obs × n_vars = 1004247 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human', 'cell_type_level1_corrected', 'cell_type_level2'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'neighbors', 'umap'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'counts', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'